In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.utils import to_categorical
import logging

# Set TensorFlow logging to ERROR to reduce spam
tf.get_logger().setLevel(logging.ERROR)

print("Starting Next Word Prediction Model...")

# 1. Sample Data (Corpus)
# In a real-world scenario, you would load this from a large text file.
data = """
A stitch in time saves nine
A penny saved is a penny earned
Actions speak louder than words
All that glitters is not gold
An apple a day keeps the doctor away
Beggars can't be choosers
Don't count your chickens before they hatch
Every cloud has a silver lining
Honesty is the best policy
Practice makes perfect
Where there's a will there's a way
"""

# 2. Preprocessing
# Clean and split the text into lines
corpus = data.lower().split("\n")
corpus = [line for line in corpus if line.strip()] # Remove empty lines

# Tokenization
# This process converts words into unique integer IDs
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1 # Add 1 for the 0-padding index

print(f"Total unique words in corpus: {total_words}")

# Create input sequences (n-grams)
# e.g., "a stitch in time" -> ["a", "a stitch", "a stitch in", "a stitch in time"]
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences to ensure they are all the same length
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# 3. Create Predictors (X) and Label (y)
# X = all words except the last one
# y = the last word
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encode the output label (y)
# We are converting the label (a single word index) into a vector
# with a 1 at the position of that word and 0s everywhere else.
y = to_categorical(y, num_classes=total_words)

print(f"Shape of X (predictors): {X.shape}")
print(f"Shape of y (labels): {y.shape}")

# 4. Build the LSTM Model
model = Sequential()
# Embedding layer: Converts word indices into dense vectors of a fixed size.
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
# Bidirectional LSTM layer: Improves context understanding by processing text in
# both forward and backward directions.
model.add(Bidirectional(LSTM(150)))
# Output layer: A standard fully connected layer.
# The number of neurons (total_words) matches our vocabulary size.
# 'softmax' ensures the output is a probability distribution across all words.
model.add(Dense(total_words, activation='softmax'))

# Compile the model
# 'categorical_crossentropy' is the standard loss function for multi-class classification.
# 'adam' is an efficient and popular optimizer.
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("\nModel Summary:")
model.summary()

# 5. Train the Model
# On this small dataset, 100 epochs will be very fast and will likely
# achieve 100% accuracy (overfitting), demonstrating it has "memorized" the proverbs.
print("\nStarting model training...")
# Note: Set verbose=0 to hide epoch progress if desired
history = model.fit(X, y, epochs=100, verbose=1)
print("Model training complete.")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")


# 6. Generation (Prediction) Function
def generate_text(seed_text, next_words, model, max_sequence_len, tokenizer):
    """
    Generates a sequence of text based on a seed.
    """
    output_text = seed_text
    print("\nGenerating text...")
    print(f'Seed: "{seed_text}"')

    for _ in range(next_words):
        # Tokenize the current text
        token_list = tokenizer.texts_to_sequences([output_text])[0]
        # Pad the sequence
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        # Predict the next word's probabilities
        predicted_probs = model.predict(token_list, verbose=0)[0]

        # Get the index of the word with the highest probability
        # Using np.argmax for a deterministic prediction
        predicted_index = np.argmax(predicted_probs)

        # (Optional) For more creative but less "accurate" text, you
        # could sample from the probability distribution instead of using argmax.
        # predicted_index = np.random.choice(len(predicted_probs), p=predicted_probs)

        # Find the word corresponding to the index
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        # Append the predicted word to the text
        if output_word:
            output_text += " " + output_word
        else:
            # Break if we can't find a word (shouldn't happen)
            break

    return output_text

# 7. Use the Model
print("\n--- Text Generation Examples ---")

seed_1 = "a penny"
generated_text_1 = generate_text(seed_1, 4, model, max_sequence_len, tokenizer)
print(f"Result: {generated_text_1}")

print("--------------------")

seed_2 = "actions speak"
generated_text_2 = generate_text(seed_2, 3, model, max_sequence_len, tokenizer)
print(f"Result: {generated_text_2}")

print("--------------------")

seed_3 = "an apple a day"
generated_text_3 = generate_text(seed_3, 4, model, max_sequence_len, tokenizer)
print(f"Result: {generated_text_3}")

print("\nScript finished.")

Starting Next Word Prediction Model...
Total unique words in corpus: 54
Shape of X (predictors): (53, 7)
Shape of y (labels): (53, 54)

Model Summary:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Starting model training...
Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.0252 - loss: 3.9883  
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.1276 - loss: 3.9702
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.1067 - loss: 3.9547
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.1067 - loss: 3.9368
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.0837 - loss: 3.9141
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.1046 - loss: 3.8780
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.0837 - loss: 3.8412
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.0941 - loss: 3.7836
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.1046 - loss: 3.6921
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.0941 - loss: 3.6747
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.0733 - loss: 3.7009
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36m